In [ ]:
# !git clone https://github.com/william-darma-wijaya/thesis-2803.git
# %cd thesis-2803

In [ ]:
# !pip install -U bitsandbytes>=0.46.1
# !pip install sentence-transformers transformers networkx pandas torch tqdm

# GraphRAG Text-to-SQL — Evaluation Pipeline

Notebook ini menjalankan dua pipeline Text-to-SQL secara berdampingan dan mengukur kualitas prediksi SQL menggunakan metrik resmi Spider benchmark:

| Pipeline | Strategi Retrieval | Pruning |
|---|---|---|
| **GraphRAG** | Two-stage: table → column (n-gram + cosine) | Ya — hanya kolom relevan masuk prompt |
| **Baseline** | Single-stage: table-level (cosine) | Tidak — semua kolom tabel masuk prompt |

**Metrik yang diukur:**
- **ESM (Exact Set Match)** — apakah semua klausa SQL secara struktural identik dengan gold SQL
- **EX (Execution Accuracy)** — apakah hasil eksekusi SQL prediksi sama dengan gold SQL
- **Schema Linking Recall/Precision** — seberapa baik retrieval memilih kolom yang relevan

Evaluasi ESM dan EX menggunakan `evaluation.py` dari Spider benchmark (dipanggil sebagai subprocess).

**Konfigurasi trial ini:** 50 sampel pertama dari Spider dev set.

## 1. Setup

Tambahkan direktori pipeline ke Python path agar modul-modul (`schema`, `retrieval`, `generation`, dll.) bisa diimport langsung. Semua logika pipeline tetap berada di modul aslinya — notebook ini hanya mengorkestrasikan.

In [ ]:
import sys
import os
import json
import re
import subprocess
from pathlib import Path

# Semua modul pipeline ada di direktori yang sama dengan notebook ini
sys.path.insert(0, str(Path(".").resolve()))

from config import PipelineConfig
from schema import load_spider_schema, build_schema_graph
from retrieval import build_schema_index
from baseline import build_table_graph, build_table_index, run_single_baseline
from few_shot import build_few_shot_index
from generation import load_model_and_tokenizer
from pipeline import run_single

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

import pandas as pd

print("Imports OK")

## 2. Konfigurasi

`PipelineConfig` adalah sumber kebenaran tunggal untuk semua parameter — path data, model, hyperparameter retrieval, dan generasi. Ubah `SAMPLE_N` untuk mengontrol jumlah sampel yang dijalankan.

Parameter yang masih **TBD** (ditentukan setelah sweep/ablation):
- `top_k_tables`, `top_k_columns`, `semantic_similarity_threshold` → tunggu hasil `sweep.py`
- `few_shot_k` → tunggu hasil ablation study

In [ ]:
cfg = PipelineConfig()

SAMPLE_N = 50  # Jumlah sampel untuk trial. Ganti ke len(dev_data) untuk full dev set.

OUTPUT_DIR = Path("eval_output")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Data path      : {cfg.data_path}")
print(f"Embedding model: {cfg.embedding_model}")
print(f"LLM model      : {cfg.llm_model}")
print(f"top_k_tables   : {cfg.top_k_tables}")
print(f"top_k_columns  : {cfg.top_k_columns}")
print(f"few_shot_k     : {cfg.few_shot_k}")
print(f"Sample N       : {SAMPLE_N}")
print(f"Output dir     : {OUTPUT_DIR.resolve()}")

## 3. Load Data (Spider Dev Set)

Spider dev set (`dev.json`) berisi 1034 pasang (question, gold SQL, db_id). Kita ambil `SAMPLE_N` sampel pertama.

Gold SQL juga ditulis ulang ke file `gold_subset.sql` dalam format yang diharapkan `evaluation.py`:
```
SELECT ...   <TAB>   db_id
```
File ini dipakai sebagai referensi saat menghitung ESM dan EX.

In [ ]:
with open(cfg.dev_json) as f:
    dev_data = json.load(f)

dev_subset = dev_data[:SAMPLE_N]

# Tulis gold SQL subset — format: {sql}\t{db_id} per baris
gold_path = OUTPUT_DIR / "gold_subset.sql"
with open(gold_path, "w", encoding="utf-8") as f:
    for item in dev_subset:
        f.write(item["query"] + "\t" + item["db_id"] + "\n")

print(f"Total dev set    : {len(dev_data)} samples")
print(f"Trial subset     : {len(dev_subset)} samples")
print(f"Gold SQL written : {gold_path}")
print()
print("Contoh sample pertama:")
ex = dev_subset[0]
print(f"  db_id    : {ex['db_id']}")
print(f"  question : {ex['question']}")
print(f"  gold SQL : {ex['query']}")

## 4. Bangun Schema Graph

Dua graph dibangun dari `tables.json` Spider dengan granularitas berbeda:

**GraphRAG graph** — node per kolom (`{db}.{table}.{col}`):
- Edge `belongs_to_pk`: kolom non-PK → PK dalam tabel yang sama
- Edge `foreign_key`: kolom FK → kolom PK di tabel lain
- Memungkinkan pruning di level kolom individu

**Baseline graph** — node per tabel (`{db}.{table}`):
- Edge `foreign_key`: tabel → tabel melalui relasi FK
- Retrieval hanya sampai level tabel, semua kolom tabel ikut masuk prompt

In [ ]:
print("Loading schema from tables.json...")
schema_df = load_spider_schema(cfg.tables_json)
print(f"Schema DataFrame: {len(schema_df)} rows")

print("\nBuilding GraphRAG column-level graph...")
graph = build_schema_graph(schema_df)
print(f"  Nodes: {graph.number_of_nodes():,}  |  Edges: {graph.number_of_edges():,}")

print("\nBuilding Baseline table-level graph...")
baseline_graph = build_table_graph(schema_df)
print(f"  Nodes: {baseline_graph.number_of_nodes():,}  |  Edges: {baseline_graph.number_of_edges():,}")

## 5. Load Models

Dua model diload sekali dan digunakan bersama oleh kedua pipeline:

| Model | Kegunaan | Catatan |
|---|---|---|
| `BAAI/bge-m3` | Schema linking + few-shot retrieval | Dipakai untuk encoding query dan schema |
| `Qwen/Qwen2.5-Coder-7B-Instruct` | SQL generation | 4-bit NF4 quantized, greedy decode |

Loading LLM membutuhkan ~5 menit dan ~6 GB VRAM. Pastikan GPU tersedia.

In [ ]:
from sentence_transformers import SentenceTransformer

print(f"Loading embedding model: {cfg.embedding_model}")
embed_model = SentenceTransformer(cfg.embedding_model)
print("Embedding model loaded.")

print(f"\nLoading LLM: {cfg.llm_model} (4-bit quantized)...")
model, tokenizer = load_model_and_tokenizer(cfg)
print("LLM loaded.")

## 6. Bangun Index

**Few-shot index** — di-build sekali dari seluruh training set (~7000 pertanyaan). Untuk setiap dev question, retriever akan mencari pertanyaan training yang paling mirip secara semantik dan menyisipkannya ke prompt sebagai contoh.

**Schema index** — di-build lazy per database (hanya ketika pertama kali ditemukan dalam loop). Menyimpan embedding per kolom dan per tabel untuk keperluan schema linking.

Build few-shot index membutuhkan beberapa menit karena mengkode seluruh training set.

In [ ]:
# Cache schema index per database — diisi lazy dalam loop
schema_cache = {}   # db_id -> SchemaIndex (untuk GraphRAG)
baseline_cache = {} # db_id -> TableSchemaIndex (untuk Baseline)

print("Building few-shot index from training set...")
few_shot_index = build_few_shot_index(cfg.train_json, embed_model)
print(f"Few-shot index built: {len(few_shot_index.examples)} training examples")

## 7. GraphRAG Inference Loop

Untuk setiap pertanyaan dalam dev subset, pipeline GraphRAG menjalankan:

1. **Stage 1** — pilih `top_k_tables` tabel kandidat via cosine similarity
2. **Stage 2** — dari kolom tabel kandidat, pilih `top_k_columns` kolom paling relevan per n-gram query
3. **Graph traversal** — cari shortest path antar kolom terpilih untuk menemukan jalur JOIN
4. **Path pruning** — buang intermediate node yang bukan PK/FK/direct hit
5. **Few-shot retrieval** — cari `few_shot_k` contoh training yang mirip
6. **Prompt + generate** — buat DDL prompt dan hasilkan SQL via Qwen2.5-Coder

Schema index untuk setiap database di-cache agar tidak dibangun ulang jika ada beberapa pertanyaan dari database yang sama.

In [ ]:
graphrag_preds = []
graphrag_metrics = []

for item in tqdm(dev_subset, desc="GraphRAG"):
    db_id = item["db_id"]

    # Lazy-build schema index per database
    if db_id not in schema_cache:
        schema_cache[db_id] = build_schema_index(graph, db_id, embed_model)

    pred_sql, recall, precision, schema_ctx, col_nodes = run_single(
        question=item["question"],
        gold_sql=item["query"],
        db_id=db_id,
        graph=graph,
        embed_model=embed_model,
        model=model,
        tokenizer=tokenizer,
        cfg=cfg,
        schema_index=schema_cache[db_id],
        few_shot_index=few_shot_index,
    )
    graphrag_preds.append(pred_sql)
    graphrag_metrics.append({"recall": recall, "precision": precision})

# Simpan prediksi ke file — satu SQL per baris
graphrag_pred_path = OUTPUT_DIR / "graphrag_predictions.txt"
graphrag_pred_path.write_text("\n".join(graphrag_preds), encoding="utf-8")

avg_recall = sum(m["recall"] for m in graphrag_metrics) / len(graphrag_metrics)
avg_prec   = sum(m["precision"] for m in graphrag_metrics) / len(graphrag_metrics)
print(f"\nGraphRAG selesai: {len(graphrag_preds)} sampel")
print(f"  Avg Schema Recall    : {avg_recall:.4f}")
print(f"  Avg Schema Precision : {avg_prec:.4f}")
print(f"  Predictions saved to : {graphrag_pred_path}")

## 8. Baseline Inference Loop

Pipeline Baseline menggunakan strategi yang lebih sederhana:

1. **Single-stage table linking** — embed query dan bandingkan langsung dengan embedding tabel (format `"table X columns Y Z..."`). Tidak ada two-stage atau n-gram matching.
2. **Table path tracing** — tambahkan tabel intermediate via FK jika ada gap konektivitas
3. **Schema context** — masukkan **semua kolom** dari setiap tabel terpilih ke prompt (tidak ada pruning)
4. **Zero-shot** — prompt tidak menyertakan few-shot examples (sesuai desain baseline)

Perbedaan utama vs GraphRAG: token count prompt biasanya lebih tinggi karena tidak ada column-level pruning.

In [ ]:
baseline_preds = []
baseline_metrics = []

for item in tqdm(dev_subset, desc="Baseline"):
    db_id = item["db_id"]

    # Lazy-build table index per database
    if db_id not in baseline_cache:
        baseline_cache[db_id] = build_table_index(baseline_graph, db_id, embed_model)

    pred_sql, recall, precision = run_single_baseline(
        question=item["question"],
        gold_sql=item["query"],
        db_id=db_id,
        graph=baseline_graph,
        embed_model=embed_model,
        model=model,
        tokenizer=tokenizer,
        cfg=cfg,
        table_index=baseline_cache[db_id],
    )
    baseline_preds.append(pred_sql)
    baseline_metrics.append({"recall": recall, "precision": precision})

# Simpan prediksi ke file
baseline_pred_path = OUTPUT_DIR / "baseline_predictions.txt"
baseline_pred_path.write_text("\n".join(baseline_preds), encoding="utf-8")

avg_recall = sum(m["recall"] for m in baseline_metrics) / len(baseline_metrics)
avg_prec   = sum(m["precision"] for m in baseline_metrics) / len(baseline_metrics)
print(f"\nBaseline selesai: {len(baseline_preds)} sampel")
print(f"  Avg Schema Recall    : {avg_recall:.4f}")
print(f"  Avg Schema Precision : {avg_prec:.4f}")
print(f"  Predictions saved to : {baseline_pred_path}")

## 9. Evaluasi: Exact Set Match & Execution Accuracy

Evaluasi resmi menggunakan `evaluation.py` dari Spider benchmark, dipanggil sebagai subprocess — persis seperti yang dilakukan `run_official_evaluation()` di `pipeline.py`.

**Perbedaan ESM vs EX:**
- **ESM** (`--etype match`) — membandingkan struktur SQL secara parse-tree. Dua SQL bisa EX=1 tapi ESM=0 jika cara penulisannya berbeda walaupun hasilnya sama.
- **EX** (`--etype exec`) — mengeksekusi kedua SQL di SQLite dan membandingkan hasilnya. Lebih realistis untuk use-case nyata, tapi bisa false-positive jika tabel kosong.

Output mencakup skor per difficulty level (easy/medium/hard/extra) dan skor keseluruhan (`all`).

In [ ]:
def run_spider_eval(pred_file: Path, gold_file: Path, db_dir: Path,
                    tables_json: Path, etype: str) -> str:
    """Panggil evaluation.py Spider dan kembalikan stdout-nya."""
    result = subprocess.run(
        [
            sys.executable, "evaluation.py",
            "--gold", str(gold_file),
            "--pred", str(pred_file),
            "--db",   str(db_dir),
            "--table", str(tables_json),
            "--etype", etype,
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f"[WARN] evaluation.py returned non-zero: {result.returncode}")
        print(result.stderr[-500:])
    return result.stdout


print("=" * 60)
print("GraphRAG — Exact Set Match")
print("=" * 60)
graphrag_esm_out = run_spider_eval(
    graphrag_pred_path, gold_path, cfg.db_dir, cfg.tables_json, "match")
print(graphrag_esm_out)

print("=" * 60)
print("GraphRAG — Execution Accuracy")
print("=" * 60)
graphrag_ex_out = run_spider_eval(
    graphrag_pred_path, gold_path, cfg.db_dir, cfg.tables_json, "exec")
print(graphrag_ex_out)

print("=" * 60)
print("Baseline — Exact Set Match")
print("=" * 60)
baseline_esm_out = run_spider_eval(
    baseline_pred_path, gold_path, cfg.db_dir, cfg.tables_json, "match")
print(baseline_esm_out)

print("=" * 60)
print("Baseline — Execution Accuracy")
print("=" * 60)
baseline_ex_out = run_spider_eval(
    baseline_pred_path, gold_path, cfg.db_dir, cfg.tables_json, "exec")
print(baseline_ex_out)

## 10. Ringkasan Hasil

Tabel berikut merangkum semua metrik untuk kedua pipeline:

- **ESM** dan **EX** diambil dari kolom `all` output Spider eval (rata-rata seluruh difficulty)
- **Schema Recall** — berapa persen kolom yang dibutuhkan gold SQL berhasil di-retrieve
- **Schema Precision** — berapa persen kolom yang di-retrieve memang dibutuhkan gold SQL

GraphRAG diharapkan memiliki **ESM/EX yang lebih tinggi** (retrieval lebih presisi) dengan **Schema Precision lebih tinggi** (sedikit kolom yang masuk = lebih tepat sasaran).

In [ ]:
def parse_score(output: str, etype: str) -> float:
    """Ekstrak skor 'all' dari stdout evaluation.py."""
    # Cari baris yang diawali 'execution' (untuk exec) atau 'exact match' (untuk match)
    key = "execution" if etype == "exec" else "exact match"
    for line in output.split("\n"):
        if line.strip().lower().startswith(key):
            nums = re.findall(r"\d+\.\d+", line)
            if nums:
                return float(nums[-1])  # kolom terakhir = 'all'
    return float("nan")


results = {
    "Pipeline": ["GraphRAG", "Baseline"],
    "ESM — all": [
        parse_score(graphrag_esm_out, "match"),
        parse_score(baseline_esm_out, "match"),
    ],
    "EX — all": [
        parse_score(graphrag_ex_out, "exec"),
        parse_score(baseline_ex_out, "exec"),
    ],
    "Schema Recall": [
        sum(m["recall"]    for m in graphrag_metrics) / len(graphrag_metrics),
        sum(m["recall"]    for m in baseline_metrics) / len(baseline_metrics),
    ],
    "Schema Precision": [
        sum(m["precision"] for m in graphrag_metrics) / len(graphrag_metrics),
        sum(m["precision"] for m in baseline_metrics) / len(baseline_metrics),
    ],
}

df = pd.DataFrame(results).set_index("Pipeline")
display(df.round(4))

print()
print(f"Total sampel dievaluasi: {SAMPLE_N}")
print(f"Output files:")
print(f"  {graphrag_pred_path}")
print(f"  {baseline_pred_path}")
print(f"  {gold_path}")